# SuperTuxKart agent — starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ml-arena/competition-baseline/blob/main/kart/agent_baseline.ipynb)

A minimal, runnable baseline for **SuperTuxKart Grand Prix**
([#169](https://ml-arena.com/viewcompetition/169)) — head-to-head 4-kart racing, ranked
by **ELO**. Unlike the random templates, this one actually **drives**: a small heuristic
that steers toward the track ahead, drifts through sharp turns, and fires power-ups.

> Open in Colab, run top to bottom. Edit your API token; `COMPETITION_ID` is already `169`.
> Beat the heuristic by improving `act`, or by training a policy offline.

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk==0.3.0"   # installs the `mlarena` package

import mlarena

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page -> API Keys)
COMPETITION_ID = 169

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Define your agent

The next cell writes **`agent.py`** with `%%writefile`; we upload the whole file so its
imports come along. The environment owns the race loop and calls `act(obs) -> action`
once per kart per step (**2 s** budget per call). `__init__` takes no arguments.

**Observation** (`obs`, a dict — all coordinates *egocentric*: `x` = lateral, `z` = forward):

- `velocity` `[3]`, `front` `[3]`, `center_path` `[3]`, `center_path_distance`, `distance_down_track`
- `max_steer_angle`, `energy`, `powerup` (int; non-zero ⇒ you hold one), `attachment`
- `paths`: nearest segments ahead, each `{"start":[3], "end":[3], "width":float}`
- `karts`: nearest opponents `[[3], ...]`; `items`: nearest `[{"pos":[3], "type":int}, ...]`

**Action** (return a dict; missing keys default to coasting):
`{"steer": -1..1, "acceleration": 0..1, "brake": 0/1, "drift": 0/1, "nitro": 0/1, "fire": 0/1, "rescue": 0/1}`

In [ ]:
%%writefile agent.py
"""Heuristic SuperTuxKart baseline: steer toward the nearest path segment ahead,
accelerate, drift on sharp turns, and fire a power-up when held."""
import math


class Agent:
    def __init__(self):
        pass

    def act(self, obs):
        paths = obs.get('paths') or []
        # Aim at the far end of the nearest path segment (x = lateral, z = forward).
        if paths:
            tx, _, tz = paths[0]['end']
        else:
            cp = obs.get('center_path', [0.0, 0.0, 1.0])
            tx, tz = cp[0], (cp[2] if len(cp) > 2 else 1.0)

        angle = math.atan2(tx, max(tz, 1e-3))          # +ve = target is to the right
        steer = max(-1.0, min(1.0, angle / (math.pi / 4)))

        sharp = abs(steer) > 0.6
        speed = sum(v * v for v in obs.get('velocity', [0, 0, 0])) ** 0.5

        # TODO: beat this heuristic — use karts/items, or load a trained policy.
        return {
            'acceleration': 0.6 if sharp else 1.0,
            'steer': steer,
            'brake': 0,
            'drift': 1 if sharp and speed > 5 else 0,
            'nitro': 1 if abs(steer) < 0.2 else 0,
            'fire': 1 if int(obs.get('powerup', 0)) != 0 else 0,
            'rescue': 0,
        }

## 2. (optional) Try it locally

Feed a tiny fake observation (track curving slightly right) and inspect the action.

In [ ]:
from agent import Agent

obs = {
    'velocity': [0.0, 0.0, 6.0],
    'paths': [{'start': [0.0, 0.0, 0.0], 'end': [0.4, 0.0, 5.0], 'width': 8.0}],
    'center_path': [0.0, 0.0, 1.0], 'powerup': 1,
}
print(Agent().act(obs))

## 3. Submit

Uploads `agent.py`, deploys it, and streams status until the race is scored.

In [ ]:
# The heuristic needs no ML framework; the default runtime is fine.
result = client.submit(competition_id=COMPETITION_ID, files=["agent.py"])
print(result)

for line in client.tail_logs(COMPETITION_ID, result["attache_agent_id"]):
    print(line)

## 4. Leaderboard

In [ ]:
client.leaderboard(COMPETITION_ID)

## 5. Where to go from here

- **Use more of the observation.** The heuristic only follows `paths[0]`. Look further
  ahead (average several segments), avoid `karts`, grab useful `items`, and time `nitro`
  on straights and `drift` through corners more precisely.
- **Manage power-ups.** Fire offensively/defensively based on nearby `karts`, and use
  `rescue` when stuck.
- **Train a policy.** Learn a controller offline (e.g. PPO on the pystk2 observation),
  save the weights, upload them next to `agent.py`, and pick a GPU framework runtime:
  `client.submit(COMPETITION_ID, files=["agent.py", "policy.pt"], runtime={"language": "python", "framework": "torch"})`.
- Ranking is **ELO** from head-to-head races — consistent finishing beats occasional
  brilliance.